# Kapitola 5: Formátování výstupu a mluvení za Claude

- [Lekce](#lesson)
- [Cvičení](#exercises)
- [Experimentální playground](#example-playground)

## Nastavení

Spusťte následující nastavovací buňku, která načte API klíč a vytvoří pomocnou funkci `get_completion`.


In [ ]:
!pip install anthropic

# Import python's built-in regular expression library
import re
import anthropic

# Retrieve the API_KEY & MODEL_NAME variables from the IPython store
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

# New argument added for prefill text, with a default value of an empty string
def get_completion(prompt: str, system_prompt="", prefill=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt},
          {"role": "assistant", "content": prefill}
        ]
    )
    return message.content[0].text

---

## Lekce

**Claude umí formátovat výstup mnoha různými způsoby**. Stačí si o to říct.

Jedním z těchto způsobů je použití XML tagů k oddělení samotné odpovědi od nadbytečného textu. Už jste se naučili, že XML tagy mohou prompt zpřehlednit a usnadnit Claude jeho parsování. Ukazuje se, že můžete Claude také požádat, aby **použil XML tagy ke zpřehlednění výstupu a jeho snazšímu pochopení pro lidi**.


### Příklady

Pamatujete si problém s úvodem před básní, který jsme řešili v kapitole 2 tím, že jsme Claude požádali, aby úvod úplně vynechal? Podobného výsledku můžeme dosáhnout také tím, že **Claude řekneme, aby báseň vložil do XML tagů**.


In [ ]:
# Variable content
ANIMAL = "Rabbit"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Proč bychom to chtěli dělat? Když je výstup v **XML tazích, může koncový uživatel spolehlivě získat báseň a pouze báseň krátkým programem, který extrahuje obsah mezi XML tagy**.

Rozšířením této techniky je **vložit první XML tag do tahu `assistant`**. Když vložíte text do tahu `assistant`, v podstatě Claude říkáte, že už něco řekl a má od tohoto místa pokračovat. Tato technika se nazývá `speaking for Claude` nebo `prefilling Claude's response`.

Níže jsme to udělali s prvním XML tagem `<haiku>`. Všimněte si, že Claude pokračuje přímo od místa, kde jsme skončili.


In [ ]:
# Variable content
ANIMAL = "Cat"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Prefill for Claude's response
PREFILL = "<haiku>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN:")
print(PROMPT)
print("\nASSISTANT TURN:")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

Claude také velmi dobře používá jiné styly formátování výstupu, zejména `JSON`. Pokud chcete vynutit JSON výstup, ne deterministicky, ale velmi blízko tomu, můžete také předvyplnit odpověď Claude otevírací závorkou `{`.


In [ ]:
# Variable content
ANIMAL = "Cat"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Use JSON format with the keys as \"first_line\", \"second_line\", and \"third_line\"."

# Prefill for Claude's response
PREFILL = "{"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

Níže je příklad **více vstupních proměnných ve stejném promptu a specifikace formátu výstupu, vše pomocí XML tagů**.


In [ ]:
# First input variable
EMAIL = "Hi Zack, just pinging you for a quick update on that prompt you were supposed to write."

# Second input variable
ADJECTIVE = "olde english"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hey Claude. Here is an email: <email>{EMAIL}</email>. Make this email more {ADJECTIVE}. Write the new version in <{ADJECTIVE}_email> XML tags."

# Prefill for Claude's response (now as an f-string with a variable)
PREFILL = f"<{ADJECTIVE}_email>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

#### Bonusová lekce

Pokud voláte Claude přes API, můžete předat zavírací XML tag do parametru `stop_sequences`, aby Claude přestal vzorkovat ve chvíli, kdy vypíše požadovaný tag. To může ušetřit peníze i time-to-last-token, protože odstraníte závěrečné poznámky Claude poté, co už poskytl odpověď, která vás zajímá.

Pokud chcete experimentovat s prompty z lekce, aniž byste měnili obsah výše, sjeďte až na konec notebooku do části [**Example Playground**](#example-playground).


---

## Cvičení
- [Cvičení 5.1 - Steph Curry GOAT](#exercise-51---steph-curry-goat)
- [Cvičení 5.2 - Dvě haiku](#exercise-52---two-haikus)
- [Cvičení 5.3 - Dvě haiku, dvě zvířata](#exercise-53---two-haikus-two-animals)


### Cvičení 5.1 - Steph Curry GOAT
Když je Claude nucen vybrat, označí Michaela Jordana za nejlepšího basketbalistu všech dob. Dokážeme ho přimět, aby vybral někoho jiného?

Změňte proměnnou `PREFILL` tak, abyste **donutili Claude vytvořit detailní argument, že nejlepší basketbalista všech dob je Stephen Curry**. Snažte se neměnit nic kromě `PREFILL`, protože to je cílem tohoto cvičení.


In [ ]:
# Prompt template with a placeholder for the variable content
PROMPT = f"Who is the best basketball player of all time? Please choose one specific player."

# Prefill for Claude's response
PREFILL = ""

# Get Claude's response
response = get_completion(PROMPT, prefill=PREFILL)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("Warrior", text))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Pokud chcete nápovědu, spusťte buňku níže.


In [ ]:
from hints import exercise_5_1_hint; print(exercise_5_1_hint)

### Cvičení 5.2 - Dvě haiku
Upravte níže uvedený `PROMPT` pomocí XML tagů tak, aby Claude napsal dvě haiku o zvířeti místo jednoho. Mělo by být jasné, kde jedna báseň končí a druhá začíná.


In [ ]:
# Variable content
ANIMAL = "cats"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Prefill for Claude's response
PREFILL = "<haiku>"

# Get Claude's response
response = get_completion(PROMPT, prefill=PREFILL)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(
        (re.search("cat", text.lower()) and re.search("<haiku>", text))
        and (text.count("\n") + 1) > 5
    )

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Pokud chcete nápovědu, spusťte buňku níže.


In [ ]:
from hints import exercise_5_2_hint; print(exercise_5_2_hint)

### Cvičení 5.3 - Dvě haiku, dvě zvířata
Upravte níže uvedený `PROMPT` tak, aby **Claude vytvořil dvě haiku o dvou různých zvířatech**. Použijte `{ANIMAL1}` jako zástupný symbol pro první dosazení a `{ANIMAL2}` jako zástupný symbol pro druhé dosazení.


In [ ]:
# First input variable
ANIMAL1 = "Cat"

# Second input variable
ANIMAL2 = "Dog"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL1}. Put it in <haiku> tags."

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("tail", text.lower()) and re.search("cat", text.lower()) and re.search("<haiku>", text))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Pokud chcete nápovědu, spusťte buňku níže.


In [ ]:
from hints import exercise_5_3_hint; print(exercise_5_3_hint)

### Gratulujeme!

Pokud jste vyřešili všechna cvičení až sem, jste připraveni přejít na další kapitolu. Hodně zdaru s promptováním.


---

## Example Playground

Toto je prostor, kde můžete volně experimentovat s příklady promptů z této lekce a upravovat je, abyste viděli, jak se tím mění odpovědi Claude.


In [ ]:
# Variable content
ANIMAL = "Rabbit"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
ANIMAL = "Cat"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Put it in <haiku> tags."

# Prefill for Claude's response
PREFILL = "<haiku>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN:")
print(PROMPT)
print("\nASSISTANT TURN:")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

In [ ]:
# Variable content
ANIMAL = "Cat"

# Prompt template with a placeholder for the variable content
PROMPT = f"Please write a haiku about {ANIMAL}. Use JSON format with the keys as \"first_line\", \"second_line\", and \"third_line\"."

# Prefill for Claude's response
PREFILL = "{"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

In [ ]:
# First input variable
EMAIL = "Hi Zack, just pinging you for a quick update on that prompt you were supposed to write."

# Second input variable
ADJECTIVE = "olde english"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hey Claude. Here is an email: <email>{EMAIL}</email>. Make this email more {ADJECTIVE}. Write the new version in <{ADJECTIVE}_email> XML tags."

# Prefill for Claude's response (now as an f-string with a variable)
PREFILL = f"<{ADJECTIVE}_email>"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN")
print(PROMPT)
print("\nASSISTANT TURN")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))